# 무역 트렌드와 산업생산의 대조 — 정본 계산기

`연구계획서.md`의 둘째 갈래. 성질별 수출 실적으로 직접 만든 품목군 월 계열에
국가데이터처 광공업생산지수(`fact_ip`)를 붙여 수출과 생산의 성장률·동조성·선후행·브리지 회귀를 잰다. 수출 척도는 명목과 코드 단위 Törnqvist 물량이고
(단가 = 명목 − 물량), 한국은행 수출물가지수(2026-09-03부터 `cache/fact_xmpi.parquet`, DB에서는 뺌)로 실질화한 값과 수출출하지수는 참고 열로만 둔다(논문에 없음).
대응표는 계획서 §VII.1 표 A(2026-09-02 확정, 논문 표 2)를 코드로 옮긴 것이다.

| 절 | 내용 |
|---|---|
| §0 | 설정. 첫 갈래 캐시, DB(읽기전용), 국면 |
| §1 | 대응표(논문 표 2)와 생산지수 구축, 코드 단위 Törnqvist 물량. 수출물가·수출출하지수는 참고 |
| §2 | 성장률. 31년·국면별 명목·단가·물량과 생산지수 (표 3~5) |
| §3 | 동조성. 물량·명목 대 생산지수 Δ12 상관, 국면별 (표 6, 그림 3) |
| §3b | 달력 설명변수와 X-13 계절조정(수출 16계열), 옛 캐시와의 대조(출력만) |
| §4 | 선후행. 계절조정 월간 변화의 교차상관과 Granger 검정 (표 7) |
| §4b | 공표 시차와 브리지 회귀. t+1월 1일·중순 정보 집합의 표본외 성적 (표 8). 10일 잠정치는 `무역트렌드_10일잠정치.ipynb`로 옮겼다 |
| §5 | 수입 투입재와 생산 (표 9) |
| §6 | 그림 1~3 |
| §7 | 검증. 논문 『관세청 수출 통계는 같은 달의 산업생산을 얼마나 먼저 알려 주는가』의 인용 수치 |

읽는 것: DB(읽기전용), 품목군 대응표 `outputs/품목군_대응표_수출.csv`, 수출물가 사본 `cache/fact_xmpi.parquet`·`dim_xmpi.parquet`, X-13 실행 파일.
다른 노트북의 산출물은 읽지 않는다(2026-09-14 독립화). §3b 끝의 대조 셀만 옛 캐시가 있으면 맞대 본다. X-13 작업 폴더는 `cache/x13_nb2`.

실행: `jupyter nbconvert --to notebook --execute --inplace 무역트렌드_산업생산대조.ipynb --ExecutePreprocessor.kernel_name=kcsdb`

## §0. 설정

In [1]:
import os, warnings
import numpy as np, pandas as pd, duckdb
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import grangercausalitytests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT: raise FileNotFoundError("KCSDB2 루트를 찾지 못했습니다")
    ROOT = up
DB   = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")
HERE = os.path.join(ROOT, "analysis", "무역 트렌드 분석")
CACHE = os.path.join(HERE, "cache"); OUT = os.path.join(HERE, "outputs"); IMG = os.path.join(HERE, "img")
MAP = pd.read_csv(os.path.join(OUT, "품목군_대응표_수출.csv"), dtype=str)
con = duckdb.connect(DB, read_only=True)

PHASES = [("P1", 1995, 2000), ("P2", 2001, 2007), ("P3", 2008, 2015), ("P4", 2016, 2019), ("P5", 2020, 2023), ("P6", 2024, 2026)]
PH_LABEL = {p: f"{a}~{b if b < 2026 else '2026.07'}" for p, a, b in PHASES}
def yrs(a, b): return (b - a) if b < 2026 else (2025 + 7/12 - a)  # 2026 끝점은 2025.08~2026.07 합. 2023년 창과의 간격은 2.58년
def phase_of(ym):
    y = ym // 100
    for p, a, b in PHASES:
        if a <= y <= b: return p
EXP15 = [s for s in MAP.group15.unique() if s != "나머지"]
# 월 금액·중량: 성질별 수출 실적(153개 코드)을 품목군 대응표로 묶는다. 다른 노트북의 캐시에 기대지 않는다(2026-09-14 독립화).
_ex = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd", how="left")
assert _ex.group15.notna().all(), "대응표에 없는 수출 코드가 있다"
_gm = _ex.groupby(["yyyymm", "group15"])[["dlr", "wgt"]].sum().unstack()
W, Q = _gm["dlr"].copy(), _gm["wgt"].copy()
for M_ in (W, Q):
    M_.index = M_.index.astype(int); M_.columns.name = None
    M_["총수출"] = M_[EXP15 + ["나머지"]].sum(axis=1)
YM0, YM1 = int(W.index.min()), int(W.index.max())
ORDER = ["총수출", "반도체 제외"] + sorted(EXP15, key=lambda s: -W.loc[202301:202512, s].sum())
print("계열", len(W.columns), "| 기간", YM0, "~", YM1, "| 품목군", len(EXP15))

계열 17 | 기간 199501 ~ 202607 | 품목군 15


## §1. 대응표와 지수 구축

표 A를 코드로 둔다. 생산지수는 소분류 둘을 묶을 때 국가데이터처의 올해 가중치로, 수출물가지수는 한국은행의 2020년 가중치로
가중평균한다. 상위 항목에서 하위 하나를 빼는 자리(반도체 제외, 화공품에서 의약품)는 $(w_A I_A - w_B I_B)/(w_A - w_B)$로 한다.
연쇄 라스파이레스 지수를 고정 가중치로 다시 묶는 것이라 근사다. 의약품 수출물가지수는 2015.12부터라 그 전의 화공품은 화학제품 전체를 쓴다.

In [2]:
# group15 → (생산지수 더할 코드, 뺄 코드, 수출물가 더할 코드, 뺄 코드, 수출출하지수 코드(있을 때만))
MAP2 = {
    "총수출":     (["0"],            [],       ["*AA"],                 [],          ["0"]),
    "반도체 제외": ([],               [],       [],                      [],          ["0"]),   # 아래에서 따로 만든다
    "반도체":     (["C261"],         [],       ["30911AA"],             [],          ["C261"]),
    "승용차":     (["C301"],         [],       ["312111AA"],            [],          []),
    "자동차부품":  (["C303"],         [],       ["31213AA"],             [],          []),
    "석유제품":    (["C192"],         [],       ["3041AA"],              [],          ["C19"]),
    "철강제품":    (["C24"],          [],       ["307AA"],               [],          ["C24"]),
    "선박":       (["C311"],         [],       [],                      [],          ["C31"]),
    "무선통신기기": (["C264"],         [],       ["30951AA"],             [],          ["C264"]),
    "컴퓨터주변기기": (["C263"],       [],       ["3094AA"],              [],          ["C263"]),
    "정밀기기":    (["C27"],          [],       ["3096AA"],              [],          ["C27"]),
    "가전제품":    (["C285", "C265"], [],       ["31015AA", "30952AA"],  [],          []),
    "일반기계":    (["C29"],          [],       ["311AA"],               [],          ["C29"]),
    "화공품":     (["C20", "C22"],   [],       ["305AA"],               ["3054AA"],  ["C20", "C22"]),
    "의약품":     (["C212"],         [],       ["305411AA"],            [],          ["C21"]),
    "제조장비":    (["C292"],         [],       ["31124AA"],             [],          []),
    "이차전지":    (["C282"],         [],       ["310131AA"],            [],          []),
}
assert set(EXP15) <= set(MAP2), set(EXP15) - set(MAP2)

# 생산지수: 지표별 (yyyymm × ksic)
IPraw = con.execute("SELECT yyyymm, ksic, measure, value FROM fact_ip").df()
IP = {m: d.pivot(index="yyyymm", columns="ksic", values="value") for m, d in IPraw.groupby("measure")}
KW = con.execute("SELECT ksic, wgt FROM dim_ksic").df().set_index("ksic").wgt.to_dict()
KN = con.execute("SELECT ksic, name_ko FROM dim_ksic").df().set_index("ksic").name_ko.to_dict()
# 수출물가지수 달러기준: (yyyymm × item_cd)
# 수출물가지수는 2026-09-03에 DB에서 뺐다(참고 계산에만 쓰므로). 뺄 때 떠 둔 cache/*.parquet에서 읽는다.
XP = duckdb.sql(f"SELECT yyyymm, item_cd, value FROM '{CACHE.replace(chr(92), '/')}/fact_xmpi.parquet' WHERE imexp='수출' AND basis='D'").df() \
        .pivot(index="yyyymm", columns="item_cd", values="value")
XW = duckdb.sql(f"SELECT item_cd, wgt FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").wgt.to_dict()
XN = duckdb.sql(f"SELECT item_cd, name_ko FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").name_ko.to_dict()

def agg_idx(P, wd, plus, minus):
    """고정 가중 합성. 뺄 항목이 결측인 달은 더할 항목만으로 둔다(의약품 2015.12 이전)."""
    if not plus: return pd.Series(np.nan, index=P.index)
    num = sum(wd[c] * P[c] for c in plus); den = sum(wd[c] for c in plus)
    if minus:
        sub = sum(wd[c] * P[c] for c in minus); dsub = sum(wd[c] for c in minus)
        out = (num - sub) / (den - dsub)
        return out.where(sub.notna(), num / den)
    return num / den

IPX = pd.DataFrame({g: agg_idx(IP["prod"],    KW, a, b) for g, (a, b, _, _, _) in MAP2.items()}).loc[YM0:YM1]
IPS = pd.DataFrame({g: agg_idx(IP["prod_sa"], KW, a, b) for g, (a, b, _, _, _) in MAP2.items()}).loc[YM0:YM1]
PX  = pd.DataFrame({g: agg_idx(XP, XW, c, d)           for g, (_, _, c, d, _) in MAP2.items()}).loc[YM0:YM1]
XS  = pd.DataFrame({g: agg_idx(IP["ship_exp"], KW, e, []) for g, (_, _, _, _, e) in MAP2.items()}).loc[YM0:YM1]
for df in (IPX, IPS, PX, XS): df.index = df.index.astype(int)
W["반도체 제외"] = W["총수출"] - W["반도체"]; Q["반도체 제외"] = Q["총수출"] - Q["반도체"]

# 반도체 제외 생산지수: 총지수에서 C261을 빼는 대신, C261을 뺀 나머지 코드(중분류 전부 + C26의 다른 소분류)를 올해
# 가중치로 묶는다. 2026년 가중치로 빼는 식 (w0·I0 − w261·I261)/(w0 − w261)은 연쇄지수에는 근사이고,
# 반도체처럼 상대 수준이 크게 움직인 항목에서는 초기 연도가 왜곡된다. 두 방식을 견줘 둔다.
DK = con.execute("SELECT ksic, level, parent FROM dim_ksic").df()
EX_ALL = [c for c in DK[DK.level == 2].ksic if c != "C26"] + ["C262", "C263", "C264", "C265"]
# 2020년 신설 중분류(C34)는 1995~2019년이 없고, 계절조정 지수는 광업 중분류 몇 개에 없다. 전 기간 있는 코드만 묶는다.
def full_codes(M): return [c for c in EX_ALL if c in M.columns and M[c].loc[YM0:YM1].notna().all()]
EX_SEMI_CODES, EX_SEMI_SA = full_codes(IP["prod"]), full_codes(IP["prod_sa"])
for lab, cds in (("원지수", EX_SEMI_CODES), ("계절조정", EX_SEMI_SA)):
    drop = [c for c in EX_ALL if c not in cds]
    print(f"반도체 제외 합성({lab}): {len(cds)}개 코드, 뺀 코드 {drop} 가중치 합 {sum(KW[c] for c in drop):.1f}/10000")
IPX["반도체 제외"] = agg_idx(IP["prod"],    KW, EX_SEMI_CODES, []).loc[YM0:YM1].values
IPS["반도체 제외"] = agg_idx(IP["prod_sa"], KW, EX_SEMI_SA,    []).loc[YM0:YM1].values
ipx_sub = agg_idx(IP["prod"], KW, ["0"], ["C261"]).loc[YM0:YM1]
_c = np.corrcoef(np.log(IPX["반도체 제외"]).diff(12).dropna(), np.log(ipx_sub).diff(12).dropna())[0, 1]
print(f"반도체 제외 생산지수: 합성 대 뺄셈의 Δ12 상관 {_c:.3f}, 2020 평균 {IPX['반도체 제외'].loc[202001:202012].mean():.1f}")

# 반도체 제외 수출물가지수: 가격이 있는 열세 품목군(선박 제외)의 연쇄 Törnqvist. 가중치는 연도별 수출액 비중.
# 2020년 고정 가중치로 총지수에서 반도체를 빼면 1990년대에 음수가 나온다 — 반도체 가격이 100배 넘게 떨어졌기 때문.
PRICED = [g for g in EXP15 if g not in ("반도체", "선박")]
def chain_tornqvist(groups):
    sh = W[groups].groupby(W.index // 100).sum(); sh = sh.div(sh.sum(axis=1), axis=0)
    dlp = np.log(PX[groups]).diff()
    yr = pd.Series(dlp.index // 100, index=dlp.index)
    w_cur = sh.reindex(yr.values).set_index(dlp.index); w_prev = sh.reindex(yr.values - 1).set_index(dlp.index)
    w_prev = w_prev.fillna(w_cur)
    wbar = 0.5 * (w_cur + w_prev)
    ok = dlp.notna()
    step = (wbar.where(ok) * dlp).sum(axis=1) / wbar.where(ok).sum(axis=1)   # 결측 품목(의약품·제조장비 초기)은 그 달 가중치에서 뺀다
    lp = step.fillna(0).cumsum()
    idx = np.exp(lp); return idx / idx.loc[202001:202012].mean() * 100
PX["반도체 제외"] = chain_tornqvist(PRICED)
px_all = chain_tornqvist(PRICED + ["반도체"])
_c = np.corrcoef(np.log(px_all).diff(12).dropna(), np.log(PX["총수출"]).diff(12).dropna())[0, 1]
print(f"열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 {_c:.3f}, 31년 연평균 {np.log(px_all.iloc[-1]/px_all.iloc[0])*100/yrs(1995,2026):.2f}% 대 "
      f"{np.log(PX['총수출'].iloc[-1]/PX['총수출'].iloc[0])*100/yrs(1995,2026):.2f}%")
XS["반도체 제외"] = agg_idx(IP["ship_exp"], KW, ["0"], ["C261"]).loc[YM0:YM1].values   # 수출출하는 소분류가 C26뿐이라 뺄셈으로
R = (W[list(MAP2)] / PX * 100)                     # 실질 수출 (2020년 달러)

# 단가와 물량은 품목군 중량을 그대로 쓰지 않고 첫 갈래와 같은 코드 단위 Törnqvist로 가른다.
# 품목군 중량은 값이 작고 무거운 코드에 끌려간다. 반도체 중량은 2024년에 한 해 만에 3분의 1이 됐는데,
# 금액 비중 1%인 기타 개별소자(44307)의 중량이 186천 톤에서 41천 톤으로 준 것이고 메모리는 5.5천 톤 그대로다.
raw = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd")
GROUP_CODES = {g: MAP.loc[MAP.group15 == g, "temper_cd"].tolist() for g in EXP15}
GROUP_CODES["총수출"] = MAP.temper_cd.tolist(); GROUP_CODES["반도체 제외"] = MAP.loc[MAP.group15 != "반도체", "temper_cd"].tolist()
def tornqvist(d0, d1):
    """d0·d1: temper_cd 인덱스, 열 dlr·wgt. 반환 (Δln V, Δln P, Δln Q), 로그 차이 그대로."""
    both = d0.join(d1, lsuffix="0", rsuffix="1", how="inner")
    both = both[(both.dlr0 > 0) & (both.dlr1 > 0) & (both.wgt0 > 0) & (both.wgt1 > 0)]
    w = 0.5 * (both.dlr0 / both.dlr0.sum() + both.dlr1 / both.dlr1.sum())
    dlp = (w * np.log((both.dlr1 / both.wgt1) / (both.dlr0 / both.wgt0))).sum()
    dlv = np.log(d1.dlr.sum() / d0.dlr.sum())
    return dlv, dlp, dlv - dlp
# 월 연쇄 물량지수 (2020=100): 달마다 앞 달과의 Törnqvist 단가 변화를 빼서 잇는다
MV = raw.pivot(index="yyyymm", columns="temper_cd", values="dlr").fillna(0); MW = raw.pivot(index="yyyymm", columns="temper_cd", values="wgt").fillna(0)
def chain_q(codes):
    v, w = MV[codes], MW[codes]
    ok = (v > 0) & (w > 0) & (v.shift(1) > 0) & (w.shift(1) > 0)
    sh = v.div(v.sum(axis=1), axis=0); wbar = 0.5 * (sh + sh.shift(1))
    dlp_i = np.log((v / w) / (v.shift(1) / w.shift(1)))
    dlp = (wbar.where(ok) * dlp_i.where(ok)).sum(axis=1) / wbar.where(ok).sum(axis=1)
    dlv = np.log(v.sum(axis=1) / v.sum(axis=1).shift(1))
    lq = (dlv - dlp).fillna(0).cumsum(); q = np.exp(lq)
    return q / q.loc[202001:202012].mean() * 100
QT = pd.DataFrame({g: chain_q(c) for g, c in GROUP_CODES.items()}).loc[YM0:YM1]
print("월 연쇄 Törnqvist 물량지수 31년 연평균(%):", {g: round(np.log(QT[g].loc[202508:202607].mean() / QT[g].loc[199501:199512].mean()) * 100 / yrs(1995, 2026), 1) for g in ["총수출", "반도체 제외", "반도체"]})

rows = []
for g, (a, b, c, d, e) in MAP2.items():
    if g == "반도체 제외":
        rows.append({"품목군": g, "생산지수": "C261을 뺀 모든 코드의 가중 합성", "수출물가지수": "열세 품목군의 연쇄 Törnqvist(수출액 가중)",
                     "수출출하지수": "0 −C261", "실질 첫 달": int(R[g].first_valid_index())}); continue
    rows.append({"품목군": g, "생산지수": "+".join(f"{k} {KN[k]}" for k in a) + ("".join(f" −{k} {KN[k]}" for k in b)),
                 "수출물가지수": "+".join(f"{k} {XN[k]}" for k in c) + ("".join(f" −{k} {XN[k]}" for k in d)) if c else "없음",
                 "수출출하지수": "+".join(e) if e else "없음",
                 "실질 첫 달": int(R[g].first_valid_index()) if R[g].notna().any() else -1})
TAB_MAP = pd.DataFrame(rows).set_index("품목군")
TAB_MAP.to_csv(os.path.join(OUT, "품목군_대응표_산업물가.csv"), encoding="utf-8-sig")
print(TAB_MAP.to_string())

반도체 제외 합성(원지수): 31개 코드, 뺀 코드 ['C34'] 가중치 합 85.8/10000


반도체 제외 합성(계절조정): 27개 코드, 뺀 코드 ['B05', 'B06', 'B07', 'C34', 'D35'] 가중치 합 693.5/10000
반도체 제외 생산지수: 합성 대 뺄셈의 Δ12 상관 0.929, 2020 평균 100.0
열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 0.950, 31년 연평균 -2.70% 대 -0.76%
월 연쇄 Törnqvist 물량지수 31년 연평균(%): {'총수출': np.float64(3.0), '반도체 제외': np.float64(3.1), '반도체': np.float64(1.0)}
                                                     생산지수                           수출물가지수   수출출하지수  실질 첫 달
품목군                                                                                                        
총수출                                                 0 총지수                          *AA 총지수        0  199501
반도체 제외                               C261을 뺀 모든 코드의 가중 합성     열세 품목군의 연쇄 Törnqvist(수출액 가중)  0 −C261  199501
반도체                                          C261 반도체 제조업                      30911AA 반도체     C261  199501
승용차                                C301 자동차용 엔진 및 자동차 제조업                     312111AA 승용차       없음  199501
자동차부품                                  C303 

## §2. 명목·단가·물량과 생산 성장률의 비교

국면 끝점(직전 국면 마지막 해 → 이 국면 마지막 해)의 연간값으로 연평균 로그 변화를 낸다. 금액·중량은 연간 합, 지수는 연간 평균이다.
2026은 2025.08~2026.07의 12개월이다. 명목 $\Delta \ln V$, 코드 단위 Törnqvist 단가 $\Delta \ln U$, 물량 $\Delta \ln Q = \Delta \ln V - \Delta \ln U$(논문 II.4)와
생산지수의 연평균 성장률을 나란히 둔다(논문 표 3~5). 수출물가 $\Delta \ln P$, 실질 $\Delta \ln V - \Delta \ln P$, 단가−물가, 수출출하지수는 참고 열이다.

In [3]:
def annual(M, how):
    """월 표를 연 표로. 2026은 2025.08~2026.07."""
    y = M.loc[:202512].groupby(M.loc[:202512].index // 100).agg(how)
    ttm = M.loc[202508:202607].agg(how).rename(2026)
    return pd.concat([y, ttm.to_frame().T])
Vy = annual(W[list(MAP2)], "sum")
Py, IPy, XSy = annual(PX, "mean"), annual(IPX, "mean"), annual(XS, "mean")
Ry = Vy / Py * 100
# 코드 단위 연간 금액·중량 (끝점용). 2026은 2025.08~2026.07의 12개월 합.
ann = raw[raw.yyyymm <= 202512].assign(year=lambda d: d.yyyymm // 100).groupby(["year", "temper_cd"])[["dlr", "wgt"]].sum().reset_index()
ttm = raw[(raw.yyyymm >= 202508) & (raw.yyyymm <= 202607)].groupby("temper_cd")[["dlr", "wgt"]].sum().reset_index().assign(year=2026)
A = pd.concat([ann, ttm], ignore_index=True).set_index(["year", "temper_cd"])
def tq(g, y0, y1):
    d0 = A.loc[y0].reindex(GROUP_CODES[g]).dropna(); d1 = A.loc[y1].reindex(GROUP_CODES[g]).dropna()
    return tornqvist(d0[["dlr", "wgt"]], d1[["dlr", "wgt"]])

rows = []
for p, a, b in PHASES:
    y0 = a - 1 if a > 1995 else 1995; y1 = b; n = yrs(y0, y1)
    for g in MAP2:
        dl = lambda T: (np.log(T.loc[y1, g] / T.loc[y0, g]) * 100 / n) if (g in T and pd.notna(T.loc[y0, g]) and pd.notna(T.loc[y1, g])) else np.nan
        dV, dP, dIP, dXS = dl(Vy), dl(Py), dl(IPy), dl(XSy)
        _, dU, dQ = [x * 100 / n for x in tq(g, y0, y1)]
        rows.append({"국면": PH_LABEL[p], "계열": g, "명목": dV, "물가": dP, "실질": dV - dP, "물량": dQ,
                     "단가": dU, "단가−물가": dU - dP, "생산지수": dIP, "수출출하지수": dXS})
TAB_REAL = pd.DataFrame(rows)
TAB_REAL.to_csv(os.path.join(OUT, "tab2_real_growth.csv"), index=False, encoding="utf-8-sig")
for col in ["실질", "물량", "생산지수", "단가−물가"]:
    print(f"\n[{col} 연평균 %]")
    print(TAB_REAL.pivot(index="계열", columns="국면", values=col).reindex(ORDER)[list(PH_LABEL.values())].round(1).to_string())


[실질 연평균 %]
국면       1995~2000  2001~2007  2008~2015  2016~2019  2020~2023  2024~2026.07
계열                                                                          
총수출           12.5       11.7        6.9        1.5        3.0           8.6
반도체 제외        10.7       10.9        6.7       -1.1        2.2           3.2
반도체           47.0       31.0       19.0       16.6       14.4          11.3
화공품           15.0        6.8        5.7        3.6        2.6          -2.9
승용차           11.7       12.4        2.4       -0.5       12.6           2.3
일반기계           7.3       14.8        5.3        2.4        1.9          -4.0
석유제품          18.0        0.6        6.5        2.3       -2.8          -0.4
철강제품           8.8        2.7        6.1       -1.3       -2.4          -4.6
선박             NaN        NaN        NaN        NaN        NaN           NaN
자동차부품         20.2       22.8        9.3       -4.2       -0.8          -7.6
무선통신기기        44.9       34.8        1.8       13.7        0.4  

In [4]:
# 31년 전체(1995→2026.07): 실질 수출 대 생산지수 — 어느 품목군이 생산보다 빨리 자랐나
n_all = yrs(1995, 2026)
full = pd.DataFrame({
    "명목": np.log(Vy.loc[2026] / Vy.loc[1995]) * 100 / n_all,
    "물가": np.log(Py.loc[2026] / Py.loc[1995]) * 100 / n_all,
    "단가": pd.Series({g: tq(g, 1995, 2026)[1] * 100 / n_all for g in MAP2}),
    "물량": pd.Series({g: tq(g, 1995, 2026)[2] * 100 / n_all for g in MAP2}),
    "생산지수": np.log(IPy.loc[2026] / IPy.loc[1995]) * 100 / n_all,
    "수출출하지수": np.log(XSy.loc[2026] / XSy.loc[1995]) * 100 / n_all})
full["실질"] = full["명목"] - full["물가"]
full["단가−물가"] = full["단가"] - full["물가"]; full["실질−생산"] = full["실질"] - full["생산지수"]; full["물량−생산"] = full["물량"] - full["생산지수"]
TAB_FULL = full.reindex(ORDER)[["명목", "물가", "실질", "단가", "물량", "단가−물가", "생산지수", "실질−생산", "물량−생산", "수출출하지수"]]
TAB_FULL.to_csv(os.path.join(OUT, "tab2_full_growth.csv"), encoding="utf-8-sig")
print(TAB_FULL.round(1).to_string())

           명목    물가    실질    단가    물량  단가−물가  생산지수  실질−생산  물량−생산  수출출하지수
총수출       6.5  -1.4   7.9   4.5   2.0    5.9   4.4    3.4   -2.5     7.3
반도체 제외    5.6  -0.8   6.4   3.0   2.5    3.9   1.9    4.5    0.6     6.2
반도체       9.5 -15.3  24.8   9.2   0.3   24.5  19.8    5.0  -19.5    20.3
화공품       6.9   0.9   6.1   1.4   5.5    0.6   1.9    4.1    3.6     3.4
승용차       7.7   0.5   7.1   1.7   5.9    1.2   2.3    4.9    3.7     NaN
일반기계      6.3   0.1   6.2   1.5   4.8    1.4   2.8    3.4    2.0     6.0
석유제품     10.2   5.5   4.7   5.1   5.1   -0.5   2.1    2.6    3.1     4.9
철강제품      5.1   2.3   2.8   1.7   3.5   -0.7   1.5    1.3    2.0     3.9
선박        5.9   NaN   NaN   7.3  -1.4    NaN   4.8    NaN   -6.2     5.1
자동차부품    10.0   0.3   9.6   1.0   9.0    0.6   5.1    4.5    3.9     NaN
무선통신기기    9.7  -8.4  18.1   8.5   1.2   16.9   3.8   14.2   -2.7     6.4
컴퓨터주변기기   6.7  -7.8  14.5  10.9  -4.2   18.6   2.9   11.6   -7.0     4.4
정밀기기      8.7  -2.2  10.9   2.5   6.2    4.7   2.9 

## §3. 동조성

전년 동월 대비 로그 차이 $\Delta_{12}$의 상관. 물량(코드 단위 Törnqvist 월 연쇄)과 명목 수출을 생산지수(원지수)와 맞대고, 국면별은 물량 대 생산지수만 낸다(논문 표 6).
실질 수출(수출물가로 나눈 것)과 수출출하지수의 상관은 참고 열이다.

In [5]:
def d12(M): return np.log(M).diff(12)
dR, dQ_, dV, dIP, dXS = d12(R), d12(QT.reindex(columns=list(MAP2))), d12(W[list(MAP2)]), d12(IPX), d12(XS)
ph = pd.Series([phase_of(x) for x in R.index], index=R.index)
rows = []
for g in MAP2:
    r = {"계열": g}
    def cc(x, y):
        j = pd.concat([x, y], axis=1).dropna()
        return (j.iloc[:, 0].corr(j.iloc[:, 1]), len(j)) if len(j) > 24 else (np.nan, len(j))
    r["물량·생산"], r["n"] = cc(dQ_[g], dIP[g])
    r["명목·생산"], _ = cc(dV[g], dIP[g])
    r["실질·생산(참고)"], _ = cc(dR[g], dIP[g])
    r["물량·수출출하(참고)"], _ = cc(dQ_[g], dXS[g])
    for p, a, b in PHASES:
        m = ph == p
        r[PH_LABEL[p]], _ = cc(dQ_[g][m], dIP[g][m])       # 국면별은 물량 대 생산
        r[PH_LABEL[p] + "(실질,참고)"], _ = cc(dR[g][m], dIP[g][m])
    rows.append(r)
TAB_SYNC2 = pd.DataFrame(rows).set_index("계열").reindex(ORDER)
TAB_SYNC2.to_csv(os.path.join(OUT, "tab2_sync.csv"), encoding="utf-8-sig")
print(TAB_SYNC2.round(2).to_string())

         물량·생산    n  명목·생산  실질·생산(참고)  물량·수출출하(참고)  1995~2000  1995~2000(실질,참고)  2001~2007  2001~2007(실질,참고)  2008~2015  2008~2015(실질,참고)  2016~2019  2016~2019(실질,참고)  2020~2023  2020~2023(실질,참고)  2024~2026.07  2024~2026.07(실질,참고)
계열                                                                                                                                                                                                                                    
총수출       0.60  367   0.64       0.66         0.70       0.05              0.39       0.74              0.68       0.82              0.80       0.64              0.70       0.74              0.77          0.79                 0.46
반도체 제외    0.57  367   0.66       0.53         0.70       0.02              0.12       0.68              0.59       0.86              0.70       0.83              0.79       0.85              0.83          0.83                 0.77
반도체       0.35  367   0.51       0.53         0.35       0.16             -0

## §3b. 달력 설명변수와 X-13 계절조정 (수출)

기술분석 노트북 §2·§3의 사양 A를 그대로 옮겨 이 노트북 안에서 계절조정한다(2026-09-14 독립화). 조업일수와 설·추석 세 창(앞 20일, 연휴 사흘, 뒤 10일)을
1995.01~2026.07의 달력월 평균으로 중심화하고 예측 구간 12개월까지 넘긴다. 로그 변환, ARIMA 자동 선택(실패하면 항공기 모형), 이상치 AO·LS, X-11(MSR).
M7 ≥ 1인 계열은 계절조정을 쓰지 않는다. 대상은 총수출과 품목군 15개이고, 반도체 제외는 §4에서 두 계절조정 수준의 차로 만든다.
수입(§5)은 같은 실행기를 쓰되 설명변수를 1997년 기준으로 따로 중심화한다. 끝 셀은 옛 캐시(기술분석 노트북 산출물)가 있으면 맞대 보기만 한다.

In [6]:
import re, subprocess, datetime as dt
from korean_lunar_calendar import KoreanLunarCalendar
X13 = r"C:\tools\x13as\x13as.exe"; assert os.path.exists(X13), "X-13 실행 파일이 없다: " + X13
XWORK = os.path.join(CACHE, "x13_nb2"); os.makedirs(XWORK, exist_ok=True)
LEAD = 12
YM_LEAD = (YM1 // 100 + (YM1 % 100 + LEAD - 1) // 12) * 100 + (YM1 % 100 + LEAD - 1) % 12 + 1
CAL_VARS = ["ln_wd", "seol_pre", "seol_dur", "seol_post", "chu_pre", "chu_dur", "chu_post"]
wd_ = con.execute("SELECT base_ym AS ym, SUM(workdays) AS workdays FROM dim_workday10d GROUP BY 1 ORDER BY 1").df().set_index("ym")
def _lunar(y, m, d):
    k = KoreanLunarCalendar(); k.setLunarDate(y, m, d, False); return dt.date.fromisoformat(k.SolarIsoFormat())
def _hol_reg(anchor, w_pre, w_post, name):
    rows = []
    for y, d0 in anchor.items():
        dur = [d0 + dt.timedelta(k) for k in (-1, 0, 1)]
        pre = [dur[0] - dt.timedelta(k) for k in range(1, w_pre + 1)]; post = [dur[-1] + dt.timedelta(k) for k in range(1, w_post + 1)]
        for kk, win in [("pre", pre), ("dur", dur), ("post", post)]:
            for d in win: rows.append({"ym": d.year * 100 + d.month, "var": f"{name}_{kk}", "w": 1 / len(win)})
    return pd.DataFrame(rows).groupby(["ym", "var"])["w"].sum().unstack(fill_value=0.0)
_H = _hol_reg({y: _lunar(y, 1, 1) for y in range(1994, 2028)}, 20, 10, "seol").add(_hol_reg({y: _lunar(y, 8, 15) for y in range(1994, 2028)}, 20, 10, "chu"), fill_value=0.0)
def cal_regressors(start):
    """달력 설명변수를 start~YM_LEAD로 자르고 start~YM1의 달력월 평균으로 중심화한다."""
    X_ = wd_.join(_H).fillna(0.0); X_["ln_wd"] = np.log(X_["workdays"]); X_ = X_.loc[start:YM_LEAD].copy(); X_["month"] = X_.index % 100
    for c in CAL_VARS:
        mm = X_.loc[:YM1].groupby("month")[c].mean(); X_[c] = X_[c] - X_["month"].map(mm)
    return X_[CAL_VARS]
Xcal_exp = cal_regressors(YM0)

def x13_run(tag, y, exog, arima=None):
    base = os.path.join(XWORK, tag); sy, sm = divmod(int(y.index.min()), 100)
    with open(base + ".dat", "w") as fh:
        for v in y.values: fh.write(f"{v:.6f}\n")
    with open(base + "_x.dat", "w") as fh:
        for row in exog.values: fh.write(" ".join(f"{v:.8f}" for v in row) + "\n")
    types = " ".join(["td"] + ["holiday"] * (exog.shape[1] - 1))
    spc = "\n".join([
        f'series{{ title="{tag}" start={sy}.{sm:02d} period=12 file="{tag}.dat" format="free" }}', 'transform{ function=log }',
        f'regression{{ user=({" ".join(exog.columns)})', f'  usertype=({types})', f'  start={sy}.{sm:02d} file="{tag}_x.dat" format="free"', '  save=(hol td) }',
        f'arima{{ model={arima} }}' if arima else 'automdl{ }', 'outlier{ types=(ao ls) }', 'estimate{ }', f'forecast{{ maxlead={LEAD} }}', 'check{ }',
        'x11{ seasonalma=msr save=(d10 d11 d12 d13) print=(d8 f2 f3) }', ""])
    with open(base + ".spc", "w") as fh: fh.write(spc)
    r = subprocess.run([X13, tag], cwd=XWORK, capture_output=True, text=True)
    if arima is None and "singular" in (r.stdout + r.stderr): return x13_run(tag, y, exog, arima="(0 1 1)(0 1 1)")
    errs = [l.strip() for l in (r.stdout + r.stderr).splitlines() if "ERROR" in l]; assert not errs, (tag, errs[:2])
    out = open(base + ".out", encoding="latin-1").read()
    g = lambda p: (lambda m: float(m.group(1)) if m else np.nan)(re.search(p, out, re.M))
    t = pd.read_csv(base + ".d11", sep=r"\s+", skiprows=2, header=None, names=["date", "v"], engine="python")
    return {"arima": (re.search(r"ARIMA Model:\s*(\([^)]*\)\s*\([^)]*\))", out) or [None, ""])[1] if re.search(r"ARIMA Model:", out) else "",
            "m7": g(r"M7\s*=\s*([\d.]+)"), "q": g(r"\*\*\* Q \(without M2\) =\s*([\d.]+)"), "beta_wd": g(r"^\s+ln_wd\s+([-\d.]+)\s"),
            "d11": pd.Series(t.v.values, index=t.date.astype(int)), "fixed": arima is not None}

X13E, _sa = {}, {}
for i, g in enumerate(["총수출"] + EXP15):
    y = W[g].dropna(); r = x13_run(f"exp{i:02d}", y, Xcal_exp.loc[y.index.min():]); X13E[g] = r
    _sa[g] = np.log(r["d11"].reindex(y.index))
SAw = pd.DataFrame(_sa)
NO_SA = {g for g, r in X13E.items() if r["m7"] >= 1}
print("수출 X-13: M7≥1(계절조정 안 씀)", sorted(NO_SA))
print(pd.DataFrame({g: {"ARIMA": r["arima"] + ("*" if r["fixed"] else ""), "β_D": r["beta_wd"], "M7": r["m7"]} for g, r in X13E.items()}).T.to_string())

수출 X-13: M7≥1(계절조정 안 씀) ['선박']
                   ARIMA     β_D     M7
총수출       (1 1 2)(0 1 1)  0.3645  0.348
석유제품      (0 1 1)(0 1 1)  0.5369  0.884
화공품       (3 1 1)(0 1 1)  0.4553  0.502
의약품       (0 1 1)(0 1 1)  0.0646   0.47
철강제품      (0 1 1)(0 1 1)  0.5899  0.544
일반기계      (0 1 1)(0 1 1)   0.491  0.366
정밀기기      (0 1 0)(0 1 1)  0.5524  0.413
제조장비      (1 1 1)(0 1 1)  0.7714  0.956
가전제품      (0 1 0)(0 1 1)  0.5462  0.405
컴퓨터주변기기   (0 1 0)(0 1 1)  0.2444  0.864
무선통신기기    (1 1 1)(1 0 1)   0.338  0.417
반도체       (3 1 1)(0 1 1)  0.1002  0.484
이차전지      (0 1 1)(0 1 1)  0.4101  0.505
승용차       (1 1 1)(0 1 1)  0.5192  0.425
자동차부품     (1 1 0)(0 1 1)  0.6751  0.557
선박       (0 1 1)(0 1 1)* -0.0052  1.076


In [7]:
# 옛 캐시와의 대조(출력만): 독립화 전에는 기술분석 노트북의 캐시를 읽었다. 파일이 있으면 맞대 보고, 차이가 있으면 원인을 볼 것.
def _cmp(label, new, path, col):
    if not os.path.exists(path): print(f"{label}: 옛 캐시 없음, 건너뜀"); return
    old = pd.read_csv(path).pivot(index="yyyymm", columns="series", values=col); old.index = old.index.astype(int)
    cols = [c for c in new.columns if c in old.columns]
    print(f"{label}: 맞댄 계열 {len(cols)} | 최대 절대 차이 {(new[cols] - old[cols].reindex(new.index)).abs().max().max():.3e}")
_cmp("월 금액", W[["총수출"] + EXP15], os.path.join(CACHE, "series_monthly.csv"), "dlr")
_cmp("월 중량", Q[["총수출"] + EXP15], os.path.join(CACHE, "series_monthly.csv"), "wgt")
_cmp("X-13 계절조정(sa_B)", SAw[["총수출"] + EXP15], os.path.join(CACHE, "series_sa.csv"), "sa_B")
_pp = os.path.join(OUT, "tab_sa_compare.csv")
if os.path.exists(_pp):
    _t = pd.read_csv(_pp, index_col=0); print("M7≥1 목록: 기술분석", sorted(set(_t.index[_t["M7"] >= 1]) & set(X13E)), "| 이 노트북", sorted(NO_SA))
else: print("tab_sa_compare.csv: 옛 캐시 없음, 건너뜀")

월 금액: 맞댄 계열 16 | 최대 절대 차이 0.000e+00
월 중량: 맞댄 계열 16 | 최대 절대 차이 0.000e+00
X-13 계절조정(sa_B): 맞댄 계열 16 | 최대 절대 차이 3.553e-15
M7≥1 목록: 기술분석 ['선박'] | 이 노트북 ['선박']


## §4. 선후행

계절조정 로그 수준의 월간 변화 $\Delta_1$로 잰다. 수출은 §3b에서 계절조정한 명목 계열(기술분석의 `sa_B`와 같은 사양)을 쓰고(실질은 참고 열),
반도체 제외는 총수출과 반도체 계절조정 수준의 차로 만든다. 생산은 국가데이터처의 계절조정 지수를 쓴다. $k>0$은 수출이 생산에 $k$개월 앞선다는 뜻이다($\mathrm{corr}(\Delta x_{t-k}, \Delta ip_t)$).
Granger 검정은 시차 3과 6으로 양방향을 본다. 계절조정을 쓰지 않는 계열(선박)은 뺀다. 강건성으로 $\Delta_{12}$ 원계열의 최대 교차상관 시차도 낸다.

In [8]:
# 계절조정 계열 SAw와 계절조정을 쓰지 않는 계열 NO_SA(M7 ≥ 1)는 §3b에서 이 노트북이 직접 만들었다.
# (2026-09-14 독립화 전에는 기술분석 노트북의 캐시 series_sa.csv·tab_sa_compare.csv를 읽었다.)
SAw["반도체 제외"] = np.log(np.exp(SAw["총수출"]) - np.exp(SAw["반도체"]))   # 근사: 두 계절조정 수준의 차
lnR_sa = SAw[[g for g in MAP2 if g in SAw]] - np.log(PX[[g for g in MAP2 if g in SAw]])
d1Rreal = lnR_sa.diff()                      # 실질(참고)
d1R = SAw[[g for g in MAP2 if g in SAw]].diff()   # 명목 계절조정 — 논문은 이것을 쓴다
d1IP = np.log(IPS).diff()

def xcorr(x, y, K=12):
    """corr(x_{t-k}, y_t), k=-K..K. k>0: x가 앞선다."""
    out = {}
    for k in range(-K, K + 1):
        j = pd.concat([x.shift(k), y], axis=1).dropna()
        out[k] = j.iloc[:, 0].corr(j.iloc[:, 1]) if len(j) > 24 else np.nan
    return pd.Series(out)
def granger_p(y, x, lag):
    """x가 y를 Granger-cause 하는가 (ssr F 검정 p값)."""
    j = pd.concat([y, x], axis=1).dropna()
    if len(j) < 60: return np.nan
    return grangercausalitytests(j.values, maxlag=[lag], verbose=False)[lag][0]["ssr_ftest"][1]

rows = []
for g in MAP2:
    if g in NO_SA or g not in d1R or d1R[g].notna().sum() < 60: continue
    xc = xcorr(d1R[g], d1IP[g]); xc12 = xcorr(dQ_[g], dIP[g]); xcr = xcorr(d1Rreal[g], d1IP[g])
    rows.append({"계열": g, "n": int(pd.concat([d1R[g], d1IP[g]], axis=1).dropna().shape[0]),
                 "동시 상관": xc[0], "최대 상관 시차": int(xc.idxmax()), "최대 상관": xc.max(),
                 "수출→생산 p(3)": granger_p(d1IP[g], d1R[g], 3), "생산→수출 p(3)": granger_p(d1R[g], d1IP[g], 3),
                 "수출→생산 p(6)": granger_p(d1IP[g], d1R[g], 6), "생산→수출 p(6)": granger_p(d1R[g], d1IP[g], 6),
                 "Δ12 최대 시차": int(xc12.idxmax()), "Δ12 최대 상관": xc12.max(),
                 "실질 동시 상관(참고)": xcr[0], "실질 수출→생산 p(3)(참고)": granger_p(d1IP[g], d1Rreal[g], 3), "실질 생산→수출 p(3)(참고)": granger_p(d1Rreal[g], d1IP[g], 3)})
TAB_LEAD = pd.DataFrame(rows).set_index("계열").reindex([g for g in ORDER if g in [r["계열"] for r in rows]])
TAB_LEAD.to_csv(os.path.join(OUT, "tab2_leadlag.csv"), encoding="utf-8-sig")
print(TAB_LEAD.round(3).to_string())

           n  동시 상관  최대 상관 시차  최대 상관  수출→생산 p(3)  생산→수출 p(3)  수출→생산 p(6)  생산→수출 p(6)  Δ12 최대 시차  Δ12 최대 상관  실질 동시 상관(참고)  실질 수출→생산 p(3)(참고)  실질 생산→수출 p(3)(참고)
계열                                                                                                                                                            
총수출      378  0.334         0  0.334       0.020       0.001       0.000       0.004          0      0.599         0.274              0.794              0.000
반도체 제외   378  0.324         0  0.324       0.002       0.057       0.000       0.038          0      0.572         0.273              0.453              0.038
반도체      378  0.394         0  0.394       0.076       0.969       0.138       0.700          0      0.347         0.409              0.491              0.479
화공품      378  0.362         0  0.362       0.001       0.666       0.000       0.235          0      0.289         0.243              0.752              0.778
승용차      378  0.680         0  0.680       0.0

In [9]:
# 총수출·반도체 제외·반도체의 교차상관 전체 모양 (±12개월)
XC = pd.DataFrame({g: xcorr(d1R[g], d1IP[g]) for g in ["총수출", "반도체 제외", "반도체"]})
XC.to_csv(os.path.join(OUT, "tab2_xcorr_profile.csv"), encoding="utf-8-sig")
print(XC.round(3).T.to_string())

          -12    -11    -10    -9     -8     -7     -6     -5     -4     -3     -2     -1     0      1      2      3      4      5      6      7      8      9      10     11     12 
총수출     0.068 -0.080  0.014  0.030 -0.049  0.058  0.055 -0.008  0.049  0.122 -0.039  0.160  0.334  0.083  0.049  0.001 -0.099 -0.058 -0.002 -0.064  0.013 -0.052 -0.045 -0.002 -0.024
반도체 제외  0.107 -0.068  0.000  0.022 -0.053  0.047  0.050 -0.015  0.046  0.118 -0.066  0.032  0.324  0.063  0.010  0.013 -0.074 -0.049 -0.025 -0.027  0.038 -0.067 -0.040  0.012  0.005
반도체    -0.096  0.012 -0.090 -0.051 -0.018 -0.051 -0.071  0.081 -0.001  0.081  0.091  0.100  0.394  0.121  0.060 -0.020 -0.047 -0.065 -0.021 -0.066 -0.006  0.094 -0.026 -0.046 -0.144


## §4b. 공표 시차와 브리지 회귀

수출과 생산이 같은 달에 움직이더라도 공표 시점이 다르면 먼저 나오는 쪽이 나중 것을 미리 짐작하게 한다. $t$월 자료의 공표는
월간 수출입 동향 $t+1$월 1일, 관세청 확정치(중량 포함) $t+1$월 중순, 광공업생산지수(산업활동동향) $t+1$월 마지막 영업일이다.
그래서 $t+1$월 1일과 중순에는 $t$월 수출과 $t-1$월 생산이 있다.
브리지 회귀 $\Delta_{12} ip_t = a + b\,\Delta_{12} ip_{t-h} + c\,\Delta_{12} x_t$로 그 정보 집합에서 $t$월 생산을 예측하고, 수출 항이 없는
기준 모형과 확장창 표본외 RMSE를 견준다. 표본외는 2010.01부터다. 기준은 전월 생산만 쓴 것과 여기에 전월 경기선행지수를 더한 것 둘이다(논문 표 8).
관세청 10일 잠정치($t$월 11일·21일)는 논문 범위 밖이라 2026-09-14에 이 절에서 걷어 내어 `무역트렌드_10일잠정치.ipynb`로 옮겼다.
**정보 집합 규칙**: 각 시점에 공표된 통계만 쓴다. $t$월 수출물가지수와 중량이 붙은 관세청 확정치는 $t+1$월 중순에 나오므로, 1일 시점의 실질화는 $t-1$월 물가로 하고 $t$월 물가·중량 물량은 중순 시점 모형으로 따로 둔다.

In [10]:
def bridge(y, x, h, start):
    # y: 목표 Δ12, x: 동월 설명변수 Δ12(없으면 None), h: 알 수 있는 생산의 시차. 표본내 R2와 확장창 표본외 RMSE(%p).
    d = pd.DataFrame({"y": y, "yl": y.shift(h)})
    if x is not None: d["x"] = x
    d = d.dropna()
    cols = ["yl"] + (["x"] if x is not None else [])
    fit = sm.OLS(d.y, sm.add_constant(d[cols])).fit()
    err = []
    for t in d.index[d.index >= start]:
        tr = d.loc[:t].iloc[:-1]
        m = sm.OLS(tr.y, sm.add_constant(tr[cols])).fit()
        xt = np.r_[1.0, d.loc[t, cols].values.astype(float)]
        err.append(d.loc[t, "y"] - float(m.params.values @ xt))
    e = np.array(err)
    return {"n": len(d), "n_oos": len(e), "R2": fit.rsquared, "coef_x": fit.params.get("x", np.nan), "t_x": fit.tvalues.get("x", np.nan),
            "RMSE": np.sqrt((e ** 2).mean()) * 100}

# t월 수출물가지수는 t+1월 중순에 나온다. t+1월 1일에 쓸 수 있는 실질화는 t-1월 물가로 나눈 것뿐이다.
dRlag = dV - d12(PX).shift(1)
dU = dV - dQ_                                      # 코드 단위 Törnqvist 단가의 Δ12 = 명목 − 물량
dVu = dV - dU.shift(1)                             # 1일 시점: 명목 t − 단가 t-1 (관세청 자료만)
rows = []
for tgt in ["총수출", "반도체 제외", "반도체"]:
    y = dIP[tgt]
    b0 = bridge(y, None, 1, 201001); b1 = bridge(y, dR[tgt], 1, 201001); b2 = bridge(y, dV[tgt], 1, 201001); b3 = bridge(y, dRlag[tgt], 1, 201001)
    b4 = bridge(y, dQ_[tgt], 1, 201001)     # 중량 Törnqvist 물량 — 물가지수 없이 관세청 자료만으로. 중량은 월간 확정치(익월 중순)에 있다
    b5 = bridge(y, dVu[tgt], 1, 201001)     # 1일 시점, 관세청 자료만: 명목 t − 전월 단가
    for lab, b in [("생산 t-1", b0), ("생산 t-1 + 명목 수출 t", b2), ("생산 t-1 + 명목 수출 t − 단가 t-1", b5), ("생산 t-1 + 물량 수출 t (중량)", b4),
                   ("생산 t-1 + 실질 수출 t (t-1월 물가)", b3), ("생산 t-1 + 실질 수출 t (t월 물가)", b1)]:
        rows.append({"생산지수": tgt, "시점": "t+1월 중순" if ("물량" in lab or "t월 물가" in lab) else "t+1월 1일", "정보": lab, "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": b["coef_x"], "t": b["t_x"]})
TAB_BRIDGE = pd.DataFrame(rows)

# 설명변수를 여러 개 받는 판. 기준 모형에 경기선행지수(t-1)를 더한 둘째 기준(논문 표 8)에 쓴다 — 선행지수는 산업활동동향과 같은 날 나온다.
# 10일 잠정치 계산(1~10일·1~20일 누적, 누적 조업일수 탄력성, 옛 표 9)은 2026-09-14에 무역트렌드_10일잠정치.ipynb로 옮겼다.
def bridge2(y, X, h, start):
    # X: 동월 설명변수 DataFrame(없으면 None). 여러 열을 허용한다.
    d = pd.DataFrame({"y": y, "yl": y.shift(h)})
    xcols = []
    if X is not None:
        for c in X.columns: d[c] = X[c]; xcols.append(c)
    d = d.dropna(); cols = ["yl"] + xcols
    fit = sm.OLS(d.y, sm.add_constant(d[cols])).fit()
    err = []
    for t in d.index[d.index >= start]:
        tr = d.loc[:t].iloc[:-1]; m = sm.OLS(tr.y, sm.add_constant(tr[cols])).fit()
        err.append(d.loc[t, "y"] - float(m.params.values @ np.r_[1.0, d.loc[t, cols].values.astype(float)]))
    e = np.array(err); xc = xcols[-1] if xcols else None
    return {"n": len(d), "n_oos": len(e), "R2": fit.rsquared, "coef_x": fit.params.get(xc, np.nan) if xc else np.nan,
            "t_x": fit.tvalues.get(xc, np.nan) if xc else np.nan, "RMSE": np.sqrt((e ** 2).mean()) * 100}

# 경기선행지수 Δ12 (DT_1C8015 A00)
cli = con.execute("SELECT yyyymm, value FROM fact_cli WHERE tbl='DT_1C8015' AND cli_cd='A00' ORDER BY 1").df().set_index("yyyymm").value
dCLI = np.log(cli).diff(12)

# 월간 모형(2010.01~ 창)에 선행지수 기준을 더한다. 논문 표 8의 둘째 기준. 선행지수 t-1은 t+1월 1일에 이미 나와 있다.
rowsc = []
for tgt in ["총수출", "반도체 제외", "반도체"]:
    y = dIP[tgt]
    X1 = pd.DataFrame({"cli": dCLI.shift(1).reindex(y.index)})
    b = bridge2(y, X1, 1, 201001)
    rowsc.append({"생산지수": tgt, "시점": "t+1월 1일", "정보": "생산 t-1 + 선행지수 t-1", "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": np.nan, "t": np.nan})
    for lab, x, when in [("명목 수출 t", dV[tgt], "t+1월 1일"), ("명목 수출 t − 단가 t-1", dVu[tgt], "t+1월 1일"), ("물량 수출 t (중량)", dQ_[tgt], "t+1월 중순")]:
        b = bridge2(y, X1.assign(x=x.reindex(y.index)), 1, 201001)
        rowsc.append({"생산지수": tgt, "시점": when, "정보": f"생산 t-1 + 선행지수 t-1 + {lab}", "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": b["coef_x"], "t": b["t_x"]})
TAB_BRIDGE = pd.concat([TAB_BRIDGE, pd.DataFrame(rowsc)], ignore_index=True)
TAB_BRIDGE.to_csv(os.path.join(OUT, "tab2_bridge.csv"), index=False, encoding="utf-8-sig")
print(TAB_BRIDGE.round(3).to_string(index=False))

  생산지수      시점                                   정보   n  표본외 n    R2  RMSE  수출 계수      t
   총수출 t+1월 1일                               생산 t-1 366    199 0.500 5.152    NaN    NaN
   총수출 t+1월 1일                     생산 t-1 + 명목 수출 t 366    199 0.602 4.463  0.196  9.631
   총수출 t+1월 1일            생산 t-1 + 명목 수출 t − 단가 t-1 366    199 0.639 4.154  0.337 11.799
   총수출 t+1월 중순                생산 t-1 + 물량 수출 t (중량) 366    199 0.660 3.747  0.385 13.087
   총수출 t+1월 1일           생산 t-1 + 실질 수출 t (t-1월 물가) 366    199 0.682 3.923  0.415 14.382
   총수출 t+1월 중순             생산 t-1 + 실질 수출 t (t월 물가) 366    199 0.656 3.967  0.401 12.845
반도체 제외 t+1월 1일                               생산 t-1 366    199 0.292 5.697    NaN    NaN
반도체 제외 t+1월 1일                     생산 t-1 + 명목 수출 t 366    199 0.509 4.289  0.268 12.671
반도체 제외 t+1월 1일            생산 t-1 + 명목 수출 t − 단가 t-1 366    199 0.493 4.462  0.354 12.000
반도체 제외 t+1월 중순                생산 t-1 + 물량 수출 t (중량) 366    199 0.533 4.015  0.414 13.688
반도체 제외 t+1월 1일       

## §5. 수입 투입재와 생산

수입 품목군은 산업생산의 투입 지표로 정한다. 덩어리 원자재(에너지·나프타·광물·철강재·비철금속)는 중량, 장비·부품·기계는 달러 금액을 쓴다.
계열마다 대응 산업의 생산지수를 두고, 수출과 같은 사양 A로 X-13ARIMA-SEATS 계절조정한 로그의 월간 변화로 교차상관($k>0$: 수입이 앞선다)과 Granger 검정을 낸다. M7 ≥ 1이면 계절조정을 쓰지 않는다(2026-09-04까지는 STL이었다).
수입은 1996년 결함 때문에 1997.01부터다.

In [11]:
DT = con.execute("SELECT temper_cd, name_ko, x2, x3 FROM dim_temper WHERE imexp='수입'").df()
def codes(prefix=None, x3=None, exclude=()):
    d = DT
    if prefix: d = d[d.temper_cd.str.startswith(prefix)]
    if x3: d = d[d.x3 == x3]
    return [c for c in d.temper_cd if c not in exclude]
IMP = {   # 이름: (부호, 단위, 대응 생산지수 코드)
    "에너지(원유·석탄·가스)": (["21101", "21102", "21103"], "wgt", ["0"]),
    "원유":                 (["21101"], "wgt", ["C192"]),
    "나프타":               (["21201"], "wgt", ["C20"]),
    "철광":                 (["22201"], "wgt", ["C241"]),
    "비철금속광":            (["22401"], "wgt", ["C242"]),
    "화공품 원자재(의약품 제외)": (codes("26", exclude=("26F01",)), "wgt", ["C20"]),
    "철강재":               (codes("27"), "wgt", ["C24"]),
    "비철금속":              (codes("28"), "wgt", ["C242"]),
    "기계류":               (codes(x3="- 기계류"), "dlr", ["0"]),
    "정밀기기":              (codes(x3="- 정밀기기"), "dlr", ["0"]),
    "반도체 제조장비":        (["31A01", "31A02"], "dlr", ["C261"]),
    "반도체":               (codes(x3="- 반도체"), "dlr", ["C261"]),
    "자동차부품":            (["33201"], "dlr", ["C301"]),
}
imp_raw = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수입' AND yyyymm >= 199701").df()
IM = pd.DataFrame({k: imp_raw[imp_raw.temper_cd.isin(cds)].groupby("yyyymm")[unit].sum() for k, (cds, unit, _) in IMP.items()})
assert (IM > 0).all().all(), IM.columns[(IM <= 0).any()]
IMIP = pd.DataFrame({k: agg_idx(IP["prod"], KW, ip, []) for k, (_, _, ip) in IMP.items()}).loc[199701:YM1]
IMIP_sa = pd.DataFrame({k: agg_idx(IP["prod_sa"], KW, ip, []) for k, (_, _, ip) in IMP.items()}).loc[199701:YM1]

# --- X-13ARIMA-SEATS(사양 A)로 수입 13계열 계절조정. 실행기(x13_run)와 달력 설명변수 함수는 §3b에 있다 ---
Xcal_imp = cal_regressors(199701)      # 수입은 1997.01부터라 그 기간의 달력월 평균으로 중심화한다

X13I, IM_sa_cols, IM_NO_SA = {}, {}, set()
for i, k in enumerate(IM):
    y = IM[k].dropna(); r = x13_run(f"imp{i:02d}", y, Xcal_imp.loc[y.index.min():]); X13I[k] = r
    if r["m7"] >= 1: IM_NO_SA.add(k)
    IM_sa_cols[k] = np.log(r["d11"].reindex(y.index))
IM_sa = pd.DataFrame(IM_sa_cols)
TAB_IMP_X13 = pd.DataFrame({k: {"ARIMA": r["arima"] + ("*" if r["fixed"] else ""), "β_D": r["beta_wd"], "M7": r["m7"], "Q": r["q"]} for k, r in X13I.items()}).T
print("수입 X-13 진단 (M7≥1이면 계절조정 안 씀):", sorted(IM_NO_SA)); print(TAB_IMP_X13.round(3).to_string())
d1M, d1MIP = IM_sa.diff(), np.log(IMIP_sa).diff()
d12M, d12MIP = d12(IM), d12(IMIP)
rows = []
for k, (cds, unit, ip) in IMP.items():
    xc12 = xcorr(d12M[k], d12MIP[k])
    if k in IM_NO_SA: xc = pd.Series({j: np.nan for j in range(-12, 13)})
    else: xc = xcorr(d1M[k], d1MIP[k])
    rows.append({"수입 품목군": k, "단위": "중량" if unit == "wgt" else "달러", "부호 수": len(cds), "M7": X13I[k]["m7"],
                 "대응 생산지수": "+".join(f"{c} {KN[c]}" for c in ip),
                 "Δ12 동시 상관": xc12[0], "Δ12 최대 시차": int(xc12.idxmax()), "Δ12 최대 상관": xc12.max(),
                 "Δ1 동시 상관": xc[0], "Δ1 최대 시차": int(xc.idxmax()) if xc.notna().any() else 0, "Δ1 최대 상관": xc.max(),
                 "수입→생산 p(6)": np.nan if k in IM_NO_SA else granger_p(d1MIP[k], d1M[k], 6),
                 "생산→수입 p(6)": np.nan if k in IM_NO_SA else granger_p(d1M[k], d1MIP[k], 6)})
TAB_IMP = pd.DataFrame(rows).set_index("수입 품목군")
TAB_IMP.to_csv(os.path.join(OUT, "tab2_imports.csv"), encoding="utf-8-sig")
print(TAB_IMP.round(3).to_string())

수입 X-13 진단 (M7≥1이면 계절조정 안 씀): ['나프타', '비철금속광', '원유', '철광']
                           ARIMA     β_D     M7     Q
에너지(원유·석탄·가스)     (0 1 1)(0 1 1)  0.2619  0.624  1.41
원유               (0 1 1)(0 1 1)*  0.0801  1.089  1.63
나프타               (0 1 1)(0 1 1)  0.2535  1.199  1.58
철광                (0 1 1)(1 0 0)  0.2562  1.929  2.02
비철금속광             (0 1 1)(0 1 1)  0.8374  1.147   1.8
화공품 원자재(의약품 제외)   (0 1 1)(0 1 1)  0.4142  0.454  0.88
철강재               (3 1 1)(0 1 1)  0.3518   0.64  0.76
비철금속              (0 1 1)(0 1 1)   0.445  0.506  0.86
기계류               (0 1 1)(0 1 1)  0.5575  0.386  0.74
정밀기기              (0 1 2)(0 1 1)  0.3999  0.272  0.46
반도체 제조장비          (2 1 0)(0 1 1)  0.4829  0.678  0.86
반도체               (0 1 1)(0 1 1)   0.335  0.405  0.66
자동차부품             (0 1 1)(0 1 1)  0.6175  0.519  0.75


                 단위  부호 수     M7                       대응 생산지수  Δ12 동시 상관  Δ12 최대 시차  Δ12 최대 상관  Δ1 동시 상관  Δ1 최대 시차  Δ1 최대 상관  수입→생산 p(6)  생산→수입 p(6)
수입 품목군                                                                                                                                               
에너지(원유·석탄·가스)    중량     3  0.624                         0 총지수      0.322         -3      0.420     0.028        -4     0.142       0.279       0.000
원유               중량     1  1.089               C192 석유 정제품 제조업      0.575          0      0.575       NaN         0       NaN         NaN         NaN
나프타              중량     1  1.199  C20 화학 물질 및 화학제품 제조업; 의약품 제외      0.136         -7      0.140       NaN         0       NaN         NaN         NaN
철광               중량     1  1.929                C241 1차 철강 제조업      0.251          0      0.251       NaN         0       NaN         NaN         NaN
비철금속광            중량     1  1.147              C242 1차 비철금속 제조업      0.057         11      0.210     

## §6. 그림 1~3

그림 1은 총수출의 명목 수출, Törnqvist 연쇄 물량, 광공업생산지수를 2020=100으로 맞춰 로그 축에 놓은 것이다(12개월 이동평균).
그림 2는 반도체에서 같은 것을 보인다. 그림 3은 물량과 생산지수 $\Delta_{12}$의 60개월 이동 상관이다(총수출, 반도체 제외, 반도체).

In [12]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "figure.dpi": 100})
def style(ax):
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.grid(True, color="0.87", linewidth=0.8); ax.set_axisbelow(True)
def tdate(idx): return pd.to_datetime(pd.Index(idx).astype(int).astype(str), format="%Y%m")
def rebase(s):
    m = s.rolling(12).mean(); return m / m.loc[202001:202012].mean() * 100

def fig_group(g, fname):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(tdate(W.index), rebase(W[g]).values, "--", color="0.55", lw=1.2, label="명목 수출(달러)")
    ax.plot(tdate(QT.index), rebase(QT[g]).values, "-", color="black", lw=1.8, label="수출 물량(Tornqvist 연쇄)")
    ax.plot(tdate(IPX.index), rebase(IPX[g]).values, "-", color="0.55", lw=1.8, label="생산지수")
    ax.set_yscale("log"); ax.set_ylabel("2020=100, 12개월 이동평균 (로그 축)")
    from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
    ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) if ax.get_ylim()[0] <= t <= ax.get_ylim()[1]]
    ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_major_formatter(FixedFormatter([str(t) for t in ticks]))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.legend(frameon=False, ncol=4, loc="upper left"); style(ax); fig.tight_layout()
    fig.savefig(os.path.join(IMG, fname), dpi=300); plt.close(fig)
fig_group("총수출", "fig_b1_exports_ip.png")
fig_group("반도체", "fig_b2_semi_ip.png")

# 그림 3: 물량 대 생산지수 Δ12의 60개월 이동 상관 — 동조성이 언제 강해지고 약해졌는지는 표로 안 보인다
ROLL = pd.DataFrame({g: dQ_[g].rolling(60).corr(dIP[g]) for g in ["총수출", "반도체 제외", "반도체"]}).dropna(how="all")   # 물량 대 생산
ROLL_R = pd.DataFrame({g: dR[g].rolling(60).corr(dIP[g]) for g in ["총수출", "반도체 제외", "반도체"]}).dropna(how="all")  # 실질(참고)
ROLL.to_csv(os.path.join(OUT, "tab2_rolling_corr.csv"), encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 4.2))
for g, ls, lw, col in [("총수출", "-", 1.8, "black"), ("반도체 제외", "--", 1.4, "black"), ("반도체", "-", 1.0, "0.55")]:
    ax.plot(tdate(ROLL.index), ROLL[g].values, ls, lw=lw, color=col, label=g)
for p, a, b in PHASES[1:]:
    ax.axvline(pd.Timestamp(f"{a}-01-01"), color="black", lw=0.8, ls=":")
ax.axhline(0, color="black", lw=0.8); ax.set_ylim(-0.6, 1.0)
ax.set_ylabel("60개월 이동 상관 (수출 물량 대 생산지수, Δ12)"); ax.legend(frameon=False, ncol=3, loc="lower left")
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig_b3_rolling_corr.png"), dpi=300); plt.close(fig)
print("그림 저장:", sorted(f for f in os.listdir(IMG) if f.startswith("fig_b")))
print(ROLL.loc[[200012, 200512, 201012, 201512, 201912, 202312, YM1]].round(2).to_string())
print("이동 상관 최소/최대 (총수출):", ROLL["총수출"].idxmin(), round(ROLL["총수출"].min(), 2), ROLL["총수출"].idxmax(), round(ROLL["총수출"].max(), 2))

그림 저장: ['fig_b1_exports_ip.png', 'fig_b2_semi_ip.png', 'fig_b3_rolling_corr.png']
         총수출  반도체 제외   반도체
yyyymm                    
200012  0.05    0.02  0.16
200512  0.75    0.63  0.42
201012  0.81    0.89  0.25
201512  0.80    0.78  0.30
201912  0.64    0.81 -0.23
202312  0.69    0.84  0.40
202607  0.64    0.71  0.33
이동 상관 최소/최대 (총수출): 200012 0.05 201406 0.91


## §7. 검증

구조 조건은 `assert`로, 본문 인용 수치는 `chk()`로 대조한다. 어긋나면 마지막에 멈춘다. 본문 수치는 집필하면서 채운다.

In [13]:
FAIL = []; N_CHK = [0]
def chk(name, calc, doc, nd=1):
    N_CHK[0] += 1
    c, d = round(float(calc), nd), round(float(doc), nd)
    if abs(c - d) > 10**(-nd) / 2 + 1e-9:
        FAIL.append(f"{name}: 계산 {c} ≠ 본문 {d}")

# 구조
assert R["총수출"].first_valid_index() == 199501 and R["총수출"].notna().all()
assert R["의약품"].first_valid_index() == 201512 and R["제조장비"].first_valid_index() == 200501
assert R["선박"].isna().all()
assert (IPX.loc[:, [g for g in MAP2]].notna().all()).all(), IPX.columns[IPX.isna().any()]
assert (PX["반도체 제외"] > 0).all() and R["반도체 제외"].notna().all()
assert all(KW[c] > 0 for a, b, _, _, _ in MAP2.values() for c in a + b)
assert all(XW[c] > 0 for _, _, c, d, _ in MAP2.values() for c in c + d)
# 반도체 제외 지수가 총지수와 반도체 사이에서 벗어나지 않는 방향으로 움직이는가 (합성의 부호 점검)
assert np.corrcoef(d12(IPX["반도체 제외"]).dropna(), d12(IPX["총수출"]).loc[d12(IPX["반도체 제외"]).dropna().index])[0, 1] > 0.8
assert TAB_REAL.shape[0] == len(PHASES) * len(MAP2) and TAB_SYNC2.shape[0] == len(MAP2)
assert len(TAB_LEAD) >= 14 and len(TAB_IMP) == len(IMP)
assert TAB_SYNC2.loc["총수출", "물량·생산"] > 0.5, TAB_SYNC2.loc["총수출", "물량·생산"]

# ---- 본문 인용 수치: 무역_트렌드_산업생산_대조.md (성질별 통계 + 생산지수) ----
PH = list(PH_LABEL.values())
def real(g, ph): return TAB_REAL[(TAB_REAL.계열 == g) & (TAB_REAL.국면 == ph)].iloc[0]
TF = TAB_FULL.copy(); TF["명목−생산"] = TF["명목"] - TF["생산지수"]
# 표 3
T3 = {"총수출": [6.5, 4.5, 2.0, 4.4, -2.5, 2.1], "반도체 제외": [5.6, 3.0, 2.5, 1.9, 0.6, 3.6], "반도체": [9.5, 9.2, 0.3, 19.8, -19.5, -10.3], "화공품": [6.9, 1.4, 5.5, 1.9, 3.6, 5.0], "승용차": [7.7, 1.7, 5.9, 2.3, 3.7, 5.4], "일반기계": [6.3, 1.5, 4.8, 2.8, 2.0, 3.5], "석유제품": [10.2, 5.1, 5.1, 2.1, 3.1, 8.1], "철강제품": [5.1, 1.7, 3.5, 1.5, 2.0, 3.6], "선박": [5.9, 7.3, -1.4, 4.8, -6.2, 1.1], "자동차부품": [10.0, 1.0, 9.0, 5.1, 3.9, 4.9], "무선통신기기": [9.7, 8.5, 1.2, 3.8, -2.7, 5.8], "컴퓨터주변기기": [6.7, 10.9, -4.2, 2.9, -7.0, 3.8], "정밀기기": [8.7, 2.5, 6.2, 2.9, 3.2, 5.8], "이차전지": [9.3, 2.4, 6.9, 7.5, -0.5, 1.9], "제조장비": [27.7, -5.9, 33.6, 2.8, 30.7, 24.9], "의약품": [12.1, 8.1, 4.0, 6.3, -2.3, 5.8], "가전제품": [-0.5, 2.3, -2.9, -0.8, -2.1, 0.2]}
for g, vals in T3.items():
    for col, v in zip(["명목", "단가", "물량", "생산지수", "물량−생산", "명목−생산"], vals): chk(f"표3 {g} {col}", TF.loc[g, col], v)
chk("개월 수", len(W), 379, 0); chk("소분류 수", con.execute("SELECT COUNT(*) FROM dim_ksic WHERE level=3").fetchone()[0], 81, 0)
chk("소분류 1995부터", con.execute("SELECT COUNT(*) FROM (SELECT f.ksic, MIN(f.yyyymm) m0 FROM fact_ip f JOIN dim_ksic d USING (ksic) WHERE d.level=3 AND f.measure='prod' GROUP BY 1) WHERE m0=199501").fetchone()[0], 76, 0)
chk("합성 대 뺄셈 상관", np.corrcoef(np.log(IPX["반도체 제외"]).diff(12).dropna(), np.log(ipx_sub).diff(12).dropna())[0, 1], 0.93, 2)
chk("반도체 중량 2023 천톤", Q.loc[202301:202312, "반도체"].sum() / 1e6, 215, 0); chk("반도체 중량 2024 천톤", Q.loc[202401:202412, "반도체"].sum() / 1e6, 71, 0)
_w = raw[raw.temper_cd == "44307"].groupby(raw.yyyymm // 100)[["dlr", "wgt"]].sum()
chk("44307 중량 2023", _w.loc[2023, "wgt"] / 1e6, 186, 0); chk("44307 중량 2024", _w.loc[2024, "wgt"] / 1e6, 41, 0)
chk("44307 금액 비중 2024 %", _w.loc[2024, "dlr"] / W.loc[202401:202412, "반도체"].sum() * 100, 1, 0)
_m = raw[raw.temper_cd == "44301"].groupby(raw.yyyymm // 100).wgt.sum(); chk("메모리 중량 2023", _m.loc[2023] / 1e6, 5.6); chk("메모리 중량 2024", _m.loc[2024] / 1e6, 5.5)
# 그림 1·2 본문 (연평균 수준의 배율)
Qy2 = annual(QT, "mean")
def ratio(T, g, y1, y0): return T.loc[y1, g] / T.loc[y0, g]
chk("생산 2008/1995", ratio(IPy, "총수출", 2008, 1995), 2.5); chk("물량 2008/1995", ratio(Qy2, "총수출", 2008, 1995), 2.1)
chk("생산 2026/2008", ratio(IPy, "총수출", 2026, 2008), 1.6); chk("물량 2026/2008", ratio(Qy2, "총수출", 2026, 2008), 1.2)
chk("생산 2026/2016", ratio(IPy, "총수출", 2026, 2016), 1.21, 2); chk("물량 2026/2016", ratio(Qy2, "총수출", 2026, 2016), 1.12, 2)
chk("명목 2008/1995", ratio(Vy, "총수출", 2008, 1995), 3.4)
chk("반도체 생산 2016/1995", ratio(IPy, "반도체", 2016, 1995), 122, 0); chk("반도체 물량 2016/1995", ratio(Qy2, "반도체", 2016, 1995), 0.8)
chk("반도체 명목 1995 억달러", Vy.loc[1995, "반도체"] / 1e8, 177, 0); chk("반도체 명목 2016 억달러", Vy.loc[2016, "반도체"] / 1e8, 627, 0)
chk("반도체 명목 1995→2016 연평균", np.log(Vy.loc[2016, "반도체"] / Vy.loc[1995, "반도체"]) * 100 / 21, 6.0)
chk("반도체 생산 1995→2016 연평균", np.log(IPy.loc[2016, "반도체"] / IPy.loc[1995, "반도체"]) * 100 / 21, 23, 0)
chk("반도체 생산 2026/2016", ratio(IPy, "반도체", 2026, 2016), 3.4); chk("반도체 물량 2026/2016", ratio(Qy2, "반도체", 2026, 2016), 1.6)
# 표 4
T4 = {("총수출", "명목"): [6.4, 11.0, 4.4, 0.7, 3.8, 14.1], ("총수출", "단가"): [-2.5, 5.7, 3.3, 1.0, 4.3, 13.5], ("총수출", "물량"): [8.9, 5.3, 1.1, -0.2, -0.4, 0.6], ("총수출", "생산지수"): [8.8, 6.4, 3.3, 1.6, 1.6, 3.2], ("반도체 제외", "명목"): [6.2, 11.7, 4.2, -1.0, 4.4, 3.9], ("반도체 제외", "단가"): [-1.9, 6.0, 2.3, -0.3, 5.6, 3.6], ("반도체 제외", "물량"): [8.0, 5.7, 1.9, -0.7, -1.2, 0.3], ("반도체 제외", "생산지수"): [2.7, 3.9, 1.9, -0.3, 0.1, 1.4], ("반도체", "명목"): [7.8, 5.8, 6.0, 10.5, 1.0, 44.8], ("반도체", "단가"): [-6.3, 3.2, 10.9, 8.2, -2.4, 39.8], ("반도체", "물량"): [14.1, 2.6, -4.9, 2.4, 3.4, 5.0], ("반도체", "생산지수"): [35.7, 23.1, 14.7, 15.8, 12.2, 13.2]}
for (g, col), vals in T4.items():
    for ph, v in zip(PH, vals): chk(f"표4 {g} {col} {ph}", real(g, ph)[col], v)
# 표 5
T5 = {"화공품": [14.4, 4.6, 6.8, 3.9, 4.6, 2.4, 3.2, 0.7, -0.3, -3.0, 0.8, -0.4], "승용차": [14.1, 5.8, 12.0, 5.6, 0.3, 1.9, -1.9, -3.8, 6.5, 0.7, 2.4, -0.4], "일반기계": [10.7, 1.7, 12.3, 6.0, 2.4, 2.8, 0.7, 1.2, -0.3, 2.0, -4.6, 0.3], "석유제품": [18.5, 7.0, 0.8, -0.1, 6.6, 2.3, 2.1, 3.8, -2.0, -0.7, 0.2, -0.5], "철강제품": [6.9, 4.0, 5.4, 3.5, 5.7, 2.2, -0.6, -0.6, -2.2, -2.1, -0.3, -2.0], "선박": [-16.5, 13.8, 3.2, 10.1, 2.8, -0.6, -8.2, -11.6, 3.5, 5.4, 5.5, 14.8], "자동차부품": [19.8, 5.2, 20.3, 8.2, 7.1, 6.0, -2.5, 0.1, -2.0, 5.6, -1.7, 0.4], "무선통신기기": [22.5, 24.4, 11.2, 10.8, -9.6, -6.6, -5.9, -12.0, -17.5, 1.9, 6.1, 5.1], "컴퓨터주변기기": [11.3, 34.7, -6.2, -4.5, -15.5, -2.8, 1.6, -6.8, -4.8, 3.3, -1.6, -7.3], "정밀기기": [13.6, 0.3, 13.3, 1.5, 9.5, 3.1, 1.5, 4.9, -9.8, 3.0, -5.7, 8.2], "이차전지": [7.3, 1.9, 8.8, 24.6, 9.0, 5.2, 5.3, 5.2, 2.5, 5.0, 4.0, -14.1], "제조장비": [158.9, -1.3, 16.0, 5.9, 16.6, 3.7, 11.0, 0.1, -6.8, 3.6, 1.2, 2.7], "의약품": [3.2, 4.7, 5.7, 8.1, 3.9, 3.0, 4.9, 6.9, 3.1, 8.3, 2.0, 10.7], "가전제품": [6.4, 2.8, -12.7, 1.5, 2.9, -0.8, -8.1, -6.6, -5.7, 0.0, -1.8, -5.6]}
for g, vals in T5.items():
    for i, ph in enumerate(PH):
        chk(f"표5 {g} 물량 {ph}", real(g, ph)["물량"], vals[2*i]); chk(f"표5 {g} 생산 {ph}", real(g, ph)["생산지수"], vals[2*i+1])
# 표 6
T6 = {"총수출": [0.60, 0.64, 0.05, 0.74, 0.82, 0.64, 0.74, 0.79], "반도체 제외": [0.57, 0.66, 0.02, 0.68, 0.86, 0.83, 0.85, 0.83], "반도체": [0.35, 0.51, 0.16, 0.47, 0.31, -0.40, 0.52, 0.07],
      "화공품": [0.29, 0.57, -0.44, 0.32, 0.51, 0.73, 0.63, 0.66], "승용차": [0.69, 0.74, 0.48, 0.79, 0.87, 0.92, 0.82, 0.91], "일반기계": [0.49, 0.58, 0.00, 0.66, 0.86, 0.51, 0.52, 0.43],
      "석유제품": [0.56, 0.39, 0.55, 0.40, 0.59, 0.21, 0.68, 0.85], "철강제품": [0.25, 0.44, -0.50, 0.44, 0.61, 0.63, 0.51, 0.50], "선박": [0.19, 0.15, 0.24, -0.10, 0.31, 0.25, 0.36, 0.20],
      "자동차부품": [0.57, 0.58, 0.46, 0.29, 0.86, 0.71, 0.79, 0.78], "무선통신기기": [0.45, 0.48, 0.39, 0.69, 0.43, -0.19, 0.46, -0.13], "컴퓨터주변기기": [0.46, 0.27, 0.24, -0.02, 0.69, 0.69, 0.50, -0.23],
      "정밀기기": [0.18, 0.21, 0.28, -0.06, 0.40, 0.56, 0.32, 0.33], "이차전지": [0.23, 0.52, 0.18, 0.12, 0.59, 0.51, -0.11, -0.43], "제조장비": [0.16, 0.16, -0.13, 0.56, 0.59, 0.51, 0.13, 0.30],
      "의약품": [0.09, 0.16, 0.20, -0.13, 0.22, 0.49, -0.09, 0.35], "가전제품": [0.44, 0.48, 0.74, 0.27, 0.53, 0.09, 0.51, 0.78]}
for g, vals in T6.items():
    for col, v in zip(["물량·생산", "명목·생산"] + PH, vals): chk(f"표6 {g} {col}", TAB_SYNC2.loc[g, col], v, 2)
chk("표6 n", TAB_SYNC2.loc["총수출", "n"], 367, 0)
# 그림 3 본문
chk("이동상관 총수출 최소", ROLL["총수출"].min(), 0.05, 2); chk("이동상관 총수출 최소 시점", ROLL["총수출"].idxmin(), 200012, 0)
chk("이동상관 총수출 최대", ROLL["총수출"].max(), 0.91, 2); chk("이동상관 총수출 최대 시점", ROLL["총수출"].idxmax(), 201406, 0); chk("이동상관 총수출 끝", ROLL["총수출"].iloc[-1], 0.64, 2)
chk("이동상관 총수출 0.5 넘는 첫 달", ROLL.index[ROLL["총수출"] > 0.5][0], 200304, 0)
chk("이동상관 총수출 2016~19 최저", ROLL.loc[201601:201912, "총수출"].min(), 0.62, 2); chk("이동상관 총수출 2016~19 최저 시점", ROLL.loc[201601:201912, "총수출"].idxmin(), 201708, 0)
chk("이동상관 반도체 제외 최소", ROLL["반도체 제외"].min(), 0.01, 2); chk("이동상관 반도체 제외 최소 시점", ROLL["반도체 제외"].idxmin(), 200201, 0)
chk("이동상관 반도체 제외 0.5 넘는 첫 달", ROLL.index[ROLL["반도체 제외"] > 0.5][0], 200407, 0)
chk("이동상관 반도체 제외 최대", ROLL["반도체 제외"].max(), 0.93, 2); chk("이동상관 반도체 제외 최대 시점", ROLL["반도체 제외"].idxmax(), 202205, 0); chk("이동상관 반도체 제외 끝", ROLL["반도체 제외"].iloc[-1], 0.71, 2)
chk("이동상관 반도체 최대", ROLL["반도체"].max(), 0.77, 2); chk("이동상관 반도체 최대 시점", ROLL["반도체"].idxmax(), 200112, 0)
chk("이동상관 반도체 최소", ROLL["반도체"].min(), -0.35, 2); chk("이동상관 반도체 최소 시점", ROLL["반도체"].idxmin(), 202008, 0); chk("이동상관 반도체 끝", ROLL["반도체"].iloc[-1], 0.33, 2)
chk("이동상관 반도체 2016~19 최저", ROLL.loc[201601:201912, "반도체"].min(), -0.23, 2); chk("이동상관 반도체 2016~19 최저 시점", ROLL.loc[201601:201912, "반도체"].idxmin(), 201912, 0)
# 표 7
T7 = {"총수출": [0.33, 0, 0.33, 0.020, 0.001, 0.000, 0.004, 0.60], "반도체 제외": [0.32, 0, 0.32, 0.002, 0.057, 0.000, 0.038, 0.57], "반도체": [0.39, 0, 0.39, 0.076, 0.969, 0.138, 0.700, 0.35], "화공품": [0.36, 0, 0.36, 0.001, 0.666, 0.000, 0.235, 0.29], "승용차": [0.68, 0, 0.68, 0.012, 0.026, 0.055, 0.035, 0.69], "일반기계": [0.17, 0, 0.17, 0.253, 0.008, 0.609, 0.020, 0.49], "석유제품": [0.29, 0, 0.29, 0.678, 0.321, 0.605, 0.437, 0.56], "철강제품": [0.18, 0, 0.18, 0.021, 0.226, 0.004, 0.221, 0.29], "자동차부품": [0.27, 0, 0.27, 0.032, 0.214, 0.064, 0.040, 0.57], "무선통신기기": [0.34, 0, 0.34, 0.073, 0.363, 0.260, 0.359, 0.45], "컴퓨터주변기기": [0.22, 0, 0.22, 0.016, 0.682, 0.099, 0.649, 0.46], "정밀기기": [0.18, 0, 0.18, 0.426, 0.703, 0.792, 0.437, 0.18], "이차전지": [0.33, 0, 0.33, 0.215, 0.179, 0.385, 0.537, 0.23], "제조장비": [0.03, -4, 0.08, 0.982, 0.294, 0.828, 0.282, 0.25], "의약품": [0.12, 0, 0.12, 0.112, 0.737, 0.214, 0.402, 0.11], "가전제품": [0.27, 0, 0.27, 0.007, 0.713, 0.062, 0.203, 0.44]}
cols7 = [("동시 상관", 2), ("최대 상관 시차", 0), ("최대 상관", 2), ("수출→생산 p(3)", 3), ("생산→수출 p(3)", 3), ("수출→생산 p(6)", 3), ("생산→수출 p(6)", 3), ("Δ12 최대 상관", 2)]
for g, vals in T7.items():
    for (col, nd), v in zip(cols7, vals): chk(f"표7 {g} {col}", TAB_LEAD.loc[g, col], v, nd)
chk("표7 n", TAB_LEAD.loc["총수출", "n"], 378, 0); chk("표7 시차 0인 계열 수", (TAB_LEAD["최대 상관 시차"] == 0).sum(), 15, 0); chk("표7 계열 수", len(TAB_LEAD), 16, 0)
chk("총수출 교차상관 동시점 밖 최대", XC["총수출"].drop(0).abs().max(), 0.16, 2)
# 표 8·9
def br(tab, tgt, info):
    r = tab[(tab.생산지수 == tgt) & (tab.정보 == info)]; assert len(r) == 1, (tgt, info); return r.iloc[0]
def red(tab, g, base, info): return (1 - br(tab, g, info).RMSE / br(tab, g, base).RMSE) * 100
B1, B2 = "생산 t-1", "생산 t-1 + 선행지수 t-1"
T8 = {("총수출", "생산 t-1"): (0.50, None, None, 5.15, None), ("총수출", "생산 t-1 + 명목 수출 t"): (0.60, 0.20, 9.6, 4.46, -13), ("총수출", "생산 t-1 + 명목 수출 t − 단가 t-1"): (0.64, 0.34, 11.8, 4.15, -19), ("총수출", "생산 t-1 + 물량 수출 t (중량)"): (0.66, 0.38, 13.1, 3.75, -27), ("총수출", "생산 t-1 + 선행지수 t-1"): (0.57, None, None, 4.92, None), ("총수출", "생산 t-1 + 선행지수 t-1 + 명목 수출 t"): (0.66, 0.19, 10.1, 4.50, -8), ("총수출", "생산 t-1 + 선행지수 t-1 + 명목 수출 t − 단가 t-1"): (0.70, 0.34, 13.0, 3.84, -22), ("총수출", "생산 t-1 + 선행지수 t-1 + 물량 수출 t (중량)"): (0.74, 0.40, 15.4, 3.32, -32), ("반도체 제외", "생산 t-1"): (0.29, None, None, 5.70, None), ("반도체 제외", "생산 t-1 + 명목 수출 t"): (0.51, 0.27, 12.7, 4.29, -25), ("반도체 제외", "생산 t-1 + 명목 수출 t − 단가 t-1"): (0.49, 0.35, 12.0, 4.46, -22), ("반도체 제외", "생산 t-1 + 물량 수출 t (중량)"): (0.53, 0.41, 13.7, 4.02, -30), ("반도체 제외", "생산 t-1 + 선행지수 t-1"): (0.38, None, None, 5.37, None), ("반도체 제외", "생산 t-1 + 선행지수 t-1 + 명목 수출 t"): (0.58, 0.26, 13.1, 4.10, -24), ("반도체 제외", "생산 t-1 + 선행지수 t-1 + 명목 수출 t − 단가 t-1"): (0.55, 0.33, 12.0, 4.17, -22), ("반도체 제외", "생산 t-1 + 선행지수 t-1 + 물량 수출 t (중량)"): (0.60, 0.40, 14.4, 3.67, -32), ("반도체", "생산 t-1"): (0.78, None, None, 9.40, None), ("반도체", "생산 t-1 + 명목 수출 t"): (0.79, 0.07, 3.8, 9.86, 5), ("반도체", "생산 t-1 + 명목 수출 t − 단가 t-1"): (0.80, 0.14, 5.9, 9.22, -2), ("반도체", "생산 t-1 + 물량 수출 t (중량)"): (0.79, 0.10, 3.9, 9.37, 0), ("반도체", "생산 t-1 + 선행지수 t-1"): (0.79, None, None, 9.35, None), ("반도체", "생산 t-1 + 선행지수 t-1 + 명목 수출 t"): (0.79, 0.06, 3.1, 9.84, 5), ("반도체", "생산 t-1 + 선행지수 t-1 + 명목 수출 t − 단가 t-1"): (0.81, 0.13, 5.7, 9.21, -1), ("반도체", "생산 t-1 + 선행지수 t-1 + 물량 수출 t (중량)"): (0.80, 0.10, 3.8, 9.34, 0)}
for (g, info), (r2, cf, tv, rm, pct) in T8.items():
    r = br(TAB_BRIDGE, g, info); chk(f"표8 {g} {info} R2", r.R2, r2, 2); chk(f"표8 {g} {info} RMSE", r.RMSE, rm, 2)
    if cf is not None:
        base = B2 if info.startswith(B2) else B1
        chk(f"표8 {g} {info} 계수", r["수출 계수"], cf, 2); chk(f"표8 {g} {info} t", r.t, tv, 1); chk(f"표8 {g} {info} 감소%", -red(TAB_BRIDGE, g, base, info), pct, 0)
chk("표8 선행지수 기준 이득 총수출", -red(TAB_BRIDGE, "총수출", B1, B2), -5, 0)
chk("표8 선행지수 기준 이득 반도체 제외", -red(TAB_BRIDGE, "반도체 제외", B1, B2), -6, 0)
chk("표8 선행지수 기준 이득 반도체", -red(TAB_BRIDGE, "반도체", B1, B2), 0, 0)
chk("표8 n", br(TAB_BRIDGE, "총수출", B1).n, 366, 0); chk("표8 표본외 n", br(TAB_BRIDGE, "총수출", B1)["표본외 n"], 199, 0)
# 10일 잠정치 문서의 검증(옛 표 9)은 2026-09-14에 무역트렌드_10일잠정치.ipynb §5로 옮겼다.

# 표 10
T10 = {"에너지(원유·석탄·가스)": [0.32, -3, 0.42, 0.03, 0.279, 0.000], "원유": [0.58, 0, 0.58, None, None, None], "나프타": [0.14, -7, 0.14, None, None, None], "철광": [0.25, 0, 0.25, None, None, None], "비철금속광": [0.06, 11, 0.21, None, None, None], "화공품 원자재(의약품 제외)": [0.54, 0, 0.54, 0.17, 0.036, 0.000], "철강재": [0.69, 0, 0.69, 0.20, 0.086, 0.000], "비철금속": [0.67, 0, 0.67, 0.01, 0.005, 0.000], "기계류": [0.60, 0, 0.60, 0.22, 0.399, 0.150], "정밀기기": [0.75, 0, 0.75, 0.30, 0.040, 0.000], "반도체 제조장비": [0.30, -3, 0.44, -0.04, 0.774, 0.000], "반도체": [0.46, -1, 0.50, 0.18, 1.000, 0.008], "자동차부품": [0.75, 0, 0.75, 0.47, 0.012, 0.342]}
cols10 = [("Δ12 동시 상관", 2), ("Δ12 최대 시차", 0), ("Δ12 최대 상관", 2), ("Δ1 동시 상관", 2), ("수입→생산 p(6)", 3), ("생산→수입 p(6)", 3)]
for g, vals in T10.items():
    for (col, nd), v in zip(cols10, vals):
        if v is not None: chk(f"표10 {g} {col}", TAB_IMP.loc[g, col], v, nd)
chk("표10 생산→수입 유의 수", (TAB_IMP["생산→수입 p(6)"] < 0.05).sum(), 7, 0); chk("표10 수입→생산 유의 수", (TAB_IMP["수입→생산 p(6)"] < 0.05).sum(), 4, 0)
chk("표10 계절조정 제외 수", (TAB_IMP["M7"] >= 1).sum(), 4, 0); assert set(TAB_IMP.index[TAB_IMP["M7"] >= 1]) == {"원유", "나프타", "철광", "비철금속광"}
for g, n in [("화공품 원자재(의약품 제외)", 18), ("기계류", 19), ("반도체", 8), ("철강재", 7), ("비철금속", 7)]: chk(f"표10 부호 수 {g}", TAB_IMP.loc[g, "부호 수"], n, 0)

# ---- 2026-09-14 추가: 본문 산문 수치와 서술의 조건 ----
# III.2 첫 문단: 2016년 이후 총수출 물량, 명목과 물량 지수가 가장 가까운 국면, 세 계열(명목·물량·생산)의 폭
for ph, v in zip(PH[-3:], (-0.2, -0.4, 0.6)): chk(f"III.2 총수출 물량 {ph}", real("총수출", ph)["물량"], v)
_span = {ph: max(real("총수출", ph)[c] for c in ("명목", "물량", "생산지수")) - min(real("총수출", ph)[c] for c in ("명목", "물량", "생산지수")) for ph in PH}
chk("III.2 2016~2019 세 계열 폭", _span[PH[3]], 1.9)
_gap = [real("총수출", ph)["생산지수"] - real("총수출", ph)["물량"] for ph in PH[-3:]]   # 2016년 이후 세 국면에서 생산이 물량보다 높은 폭
chk("III.2 2016년 이후 생산-물량 최소", min(_gap), 1.9); chk("III.2 2016년 이후 생산-물량 최대", max(_gap), 2.6)
assert min(_span, key=_span.get) == PH[3] and max(_span, key=_span.get) == PH[-1], _span
assert min(PH, key=lambda ph: abs(real("총수출", ph)["단가"])) == PH[3]
# III.2 표 5 뒤: 물량과 생산의 부호가 다른 국면 수 (표 5의 품목군 14개 — 반도체는 표 4에 있다)
_nsign = {g: sum(np.sign(real(g, ph)["물량"]) != np.sign(real(g, ph)["생산지수"]) for ph in PH) for g in EXP15 if g != "반도체"}
if sorted(g for g, n in _nsign.items() if n == 0) != ["의약품", "철강제품"]: FAIL.append(f"III.2 부호 모두 같은 품목군 {_nsign}")
if sorted(g for g, n in _nsign.items() if n == 1) != sorted(["승용차", "화공품", "무선통신기기", "이차전지"]): FAIL.append(f"III.2 부호 한 국면만 다른 품목군 {_nsign}")
if not all(np.sign(real("자동차부품", ph)["물량"]) != np.sign(real("자동차부품", ph)["생산지수"]) for ph in PH[3:]): FAIL.append("III.2 자동차부품 2016년 이후 부호")
# V 표 7: 물량 Δ12 최대 상관의 시차가 0이 아닌 계열
_lag = TAB_LEAD["Δ12 최대 시차"]
if sorted(_lag[_lag != 0].index) != sorted(["철강제품", "제조장비", "의약품"]): FAIL.append(f"표7 Δ12 시차 0이 아닌 계열 {list(_lag[_lag != 0].index)}")
for g, k in [("철강제품", -7), ("제조장비", -6), ("의약품", 5)]: chk(f"표7 Δ12 최대 시차 {g}", _lag[g], k, 0)
# VII: 자동차부품을 뺀 Δ1 동시 상관의 최댓값, 표본 시작
chk("VII 자동차부품 뺀 Δ1 최댓값", TAB_IMP["Δ1 동시 상관"].drop("자동차부품").max(), 0.30, 2)
chk("VII Δ1 첫 달", d1M.dropna(how="all").index[0], 199702, 0); chk("VII Δ12 첫 달", d12M.dropna(how="all").index[0], 199801, 0)
# II.3: 반도체 제외 생산지수에서 빠진 코드와 가중치
_drop_raw, _drop_sa = [c for c in EX_ALL if c not in EX_SEMI_CODES], [c for c in EX_ALL if c not in EX_SEMI_SA]
if _drop_raw != ["C34"]: FAIL.append(f"II.3 원지수 뺀 코드 {_drop_raw}")
if sorted(_drop_sa) != ["B05", "B06", "B07", "C34", "D35"]: FAIL.append(f"II.3 계절조정 뺀 코드 {_drop_sa}")
chk("II.3 C34 가중치 %", KW["C34"] / 100, 0.86, 2); chk("II.3 계절조정 뺀 가중치 %", sum(KW[c] for c in _drop_sa) / 100, 6.9, 1)
# IV 표 6: 국면별 첫 열은 Δ12가 1996.01부터
chk("IV 표6 Δ12 첫 달", dQ_["총수출"].dropna().index[0], 199601, 0)

print(f"검증 {N_CHK[0]}건, 어긋남 {len(FAIL)}건")
for f in FAIL: print("  ", f)
assert not FAIL

검증 854건, 어긋남 0건
